## Configurable Native-T + Seed
### RWF-2000 + UCF-Crime Fighting subset + Bus Violence

Notebook ini **hanya melakukan evaluation/inference** menggunakan `best.pt`
yang sudah selesai dilatih.

Tidak ada:
- training ulang;
- optimizer;
- scheduler;
- backpropagation;
- augmentation training;
- model selection menggunakan held-out data.

Satu notebook ini dapat dipakai untuk:

```text
T ∈ {1, 2, 4, 8}
seed ∈ {0, 42, 123}
```

Dengan hanya mengubah dua variabel:

```python
RUN_SEED = 0
NATIVE_T = 8
```

Arsitektur mengikuti notebook training MobileNetV2+TSM yang diberikan:

```text
16-frame cache
→ deterministic native-T segment-center selection
→ MobileNetV2
   + Zhang-style Method-2 TSM pada residual InvertedResidual blocks
→ per-segment logits
→ average-logit consensus
→ clip logits
```

## Perilaku khusus T=1

Pada training T=1, **wrapper TSM tetap dipasang pada residual blocks**, tetapi
`TemporalShift(n_segment=1)` menjadi **identity** karena tidak ada temporal
neighbor. Jadi evaluator T=1 tidak boleh mengganti arsitektur menjadi
MobileNetV2 tanpa wrapper TSM; wrapper tetap dibuat agar struktur checkpoint
sama dengan training, sedangkan operasi shift-nya mengembalikan input apa adanya.

Dengan demikian satu implementation evaluator tetap cukup untuk seluruh T.

Deterministic evaluation indices dari shared cache 16-frame:

```text
T=1 : [8]
T=2 : [4, 12]
T=4 : [2, 6, 10, 14]
T=8 : [1, 3, 5, 7, 9, 11, 13, 15]
```

Semua indeks 0-based.

Held-out domain:
1. RWF-2000
2. UCF-Crime Fighting subset
3. Bus Violence

Prediction CSV disimpan clip-by-clip untuk paired bootstrap dan McNemar.

In [79]:
# Tidak perlu timm untuk MobileNetV2; torchvision sudah tersedia di Colab.

In [80]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [81]:
import os
import json
import random
import platform
import warnings
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import mobilenet_v2
from torchvision.models.mobilenetv2 import InvertedResidual
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

Python      : 3.13.15
PyTorch     : 2.11.0+cu128
torchvision : 0.26.0+cu128
CUDA        : True
GPU         : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

### Hanya dua variabel yang diubah

```python
RUN_SEED = 0
NATIVE_T = 8
```

Valid:

```python
RUN_SEED ∈ {0, 42, 123}
NATIVE_T ∈ {1, 2, 4, 8}
```

Untuk mengganti **seed pada T yang sama**, cukup ubah `RUN_SEED`.

Untuk berpindah dari T=1 ke T=2/T=4/T=8, ubah `NATIVE_T`.

### Naming checkpoint yang didukung

Training lama T=8 menggunakan:

```text
MODEL_VARIANT = "mobilenet_tsm"
Zhang2022_mobilenet_tsm_SeedX_v1/
└── mobilenet_tsm/
    └── seed_X/
        └── best.pt
```

Native-T baru menggunakan pola:

```text
mobilenet_tsm_native_TX_SeedX_v1/
└── mobilenet_tsm_native_TX/
    └── seed_X/
        └── best.pt
```

Resolver mendukung kedua pola dan tetap melakukan metadata audit agar checkpoint
T/seed yang salah tidak dipakai.

In [ ]:
# ============================================================
# HANYA UBAH DUA NILAI INI
# ============================================================
RUN_SEED = 42
NATIVE_T = 8

VALID_SEEDS = (0, 42, 123)
VALID_NATIVE_T = (1, 2, 4, 8)

if RUN_SEED not in VALID_SEEDS:
    raise ValueError(
        f"RUN_SEED harus salah satu dari {VALID_SEEDS}, mendapat {RUN_SEED}."
    )

if NATIVE_T not in VALID_NATIVE_T:
    raise ValueError(
        f"NATIVE_T harus salah satu dari {VALID_NATIVE_T}, mendapat {NATIVE_T}."
    )

# ============================================================
# MODEL / CHECKPOINT
# ============================================================
MODELS_ROOT = Path(
    "/content/drive/MyDrive/IVS/ViolenceDetection/Models"
)

COMPARATOR_ROOT = (
    MODELS_ROOT
    / "Comparators"
)

USE_TSM = True
SHIFT_DIV = 8

CLASS_NAMES = [
    "NonViolence",
    "Violence",
]

NUM_CLASSES = 2

CACHE_SEQUENCE_LENGTH = 16
NUM_SEGMENTS = NATIVE_T

RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224

SEGMENT_WIDTH = (
    CACHE_SEQUENCE_LENGTH
    // NUM_SEGMENTS
)

if (
    CACHE_SEQUENCE_LENGTH
    % NUM_SEGMENTS
    != 0
):
    raise ValueError(
        f"CACHE_SEQUENCE_LENGTH={CACHE_SEQUENCE_LENGTH} "
        f"tidak habis dibagi NATIVE_T={NUM_SEGMENTS}."
    )

# ------------------------------------------------------------
# Variant naming
# ------------------------------------------------------------
# Training legacy T=8:
#   mobilenet_tsm
#
# Native-T experiments:
#   mobilenet_tsm_native_T1
#   mobilenet_tsm_native_T2
#   mobilenet_tsm_native_T4
#   dan kemungkinan mobilenet_tsm_native_T8
PRIMARY_VARIANT = (
    "mobilenet_tsm"
    if NATIVE_T == 8
    else f"mobilenet_tsm_native_T{NATIVE_T}"
)

ALTERNATE_VARIANT = (
    f"mobilenet_tsm_native_T{NATIVE_T}"
    if PRIMARY_VARIANT == "mobilenet_tsm"
    else "mobilenet_tsm"
)

# Optional manual override.
# Normalnya biarkan None.
CHECKPOINT_PATH_OVERRIDE = "/content/drive/MyDrive/mobilenet_tsm/seed_42/best.pt"


def candidate_checkpoint_paths(
    seed: int,
    native_t: int,
) -> List[Tuple[Path, str]]:
    variants = list(
        dict.fromkeys(
            [
                PRIMARY_VARIANT,
                ALTERNATE_VARIANT,
                f"mobilenet_tsm_native_T{native_t}",
            ]
        )
    )

    candidates = []

    for variant in variants:
        root_names = [
            f"{variant}_Seed{seed}_v1",
            f"Zhang2022_{variant}_Seed{seed}_v1",
            f"{variant}_Seed{seed}",
            f"Zhang2022_{variant}_Seed{seed}",
        ]

        # Legacy T=8 exact convention.
        if native_t == 8:
            root_names.extend(
                [
                    f"Zhang2022_mobilenet_tsm_Seed{seed}_v1",
                    f"mobilenet_tsm_Seed{seed}_v1",
                ]
            )

        for root_name in root_names:
            candidates.append(
                (
                    COMPARATOR_ROOT
                    / root_name
                    / variant
                    / f"seed_{seed}"
                    / "best.pt",
                    variant,
                )
            )

    # Deduplicate path while preserving first variant label.
    dedup = {}
    for path, variant in candidates:
        if path not in dedup:
            dedup[path] = variant

    return [
        (path, variant)
        for path, variant
        in dedup.items()
    ]


def resolve_checkpoint(
    seed: int,
    native_t: int,
) -> Tuple[Path, str]:
    if (
        CHECKPOINT_PATH_OVERRIDE
        is not None
    ):
        path = Path(
            CHECKPOINT_PATH_OVERRIDE
        )

        if not path.exists():
            raise FileNotFoundError(
                f"CHECKPOINT_PATH_OVERRIDE tidak ditemukan: {path}"
            )

        return (
            path,
            "manual_override",
        )

    direct_matches = [
        (path, variant)
        for path, variant
        in candidate_checkpoint_paths(
            seed,
            native_t,
        )
        if path.exists()
    ]

    # Deduplicate.
    unique = {}
    for path, variant in direct_matches:
        unique[path] = variant

    direct_matches = [
        (path, variant)
        for path, variant
        in unique.items()
    ]

    if len(
        direct_matches
    ) == 1:
        return direct_matches[0]

    if len(
        direct_matches
    ) > 1:
        raise RuntimeError(
            "Ditemukan lebih dari satu candidate checkpoint "
            f"MobileNetV2+TSM T={native_t}, seed={seed}. "
            "Agar tidak salah model, isi CHECKPOINT_PATH_OVERRIDE:\n"
            + "\n".join(
                str(path)
                for path, _
                in direct_matches
            )
        )

    # --------------------------------------------------------
    # Fallback path scan.
    # Tidak memilih secara diam-diam jika ambigu.
    # --------------------------------------------------------
    recursive_matches = []

    for path in COMPARATOR_ROOT.glob(
        f"**/seed_{seed}/best.pt"
    ):
        low = str(
            path
        ).lower()

        if "mobilenet" not in low:
            continue

        if "tsm" not in low:
            continue

        # Exclude unrelated TSM families.
        if "wang2024" in low:
            continue

        # T<8 wajib membawa native-T token.
        if native_t in (1, 2, 4):
            token_a = (
                f"native_t{native_t}"
            )
            token_b = (
                f"nativet{native_t}"
            )

            if (
                token_a not in low
                and token_b not in low
            ):
                continue

        # T=8: accept legacy mobilenet_tsm or native_T8.
        if native_t == 8:
            if any(
                token in low
                for token in [
                    "native_t1",
                    "native_t2",
                    "native_t4",
                    "nativet1",
                    "nativet2",
                    "nativet4",
                ]
            ):
                continue

        recursive_matches.append(
            path
        )

    recursive_matches = sorted(
        dict.fromkeys(
            recursive_matches
        )
    )

    if len(
        recursive_matches
    ) == 1:
        path = (
            recursive_matches[0]
        )

        resolved_variant = (
            path
            .parent
            .parent
            .name
        )

        return (
            path,
            resolved_variant,
        )

    if len(
        recursive_matches
    ) == 0:
        raise FileNotFoundError(
            "Checkpoint MobileNetV2+TSM tidak ditemukan.\n"
            f"Required: T={native_t}, seed={seed}\n"
            f"Search root: {COMPARATOR_ROOT}\n"
            "Jika struktur folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
        )

    raise RuntimeError(
        "Ditemukan beberapa checkpoint MobileNetV2+TSM yang mungkin cocok.\n"
        "Set CHECKPOINT_PATH_OVERRIDE agar eksplisit:\n"
        + "\n".join(
            str(path)
            for path
            in recursive_matches
        )
    )


CHECKPOINT_PATH, RESOLVED_VARIANT = resolve_checkpoint(
    RUN_SEED,
    NATIVE_T,
)

EVAL_OUTPUT_DIR = (
    MODELS_ROOT
    / "EffTempNet"
    / "HeldOut_Evaluation"
    / "MobileNetV2_TSM"
    / f"T{NATIVE_T}"
    / f"seed_{RUN_SEED}"
)

# ============================================================
# HELD-OUT DATASET
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/Benchmark"
)

EFFTEMPNET_MODEL_ROOT = (
    MODELS_ROOT
    / "EffTempNet"
)

# RWF cache yang sama dengan eksperimen lama.
EXISTING_CACHE_DIR = (
    EFFTEMPNET_MODEL_ROOT
    / "new_saved_dataset"
)

# Shared UCF/Bus cache dengan evaluator model lain.
HELDOUT_CACHE_DIR = (
    EFFTEMPNET_MODEL_ROOT
    / "heldout_cache_rgb64_v2"
)

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "RWF"
        ),
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "UCF": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "UCF"
        ),
        "cache_name": "UCF",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "BV": {
        "raw_dir": (
            BENCHMARK_ROOT
            / "BV"
        ),
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

NONVIOLENCE_ALIASES = (
    "NonViolence",
    "Non-Violence",
    "Non_Violence",
    "Nonviolent",
    "NonViolent",
    "NonFight",
    "Non-Fight",
    "Non_Fight",
    "Normal",
    "NormalVideos",
    "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence",
    "Violent",
    "Fight",
    "Fighting",
    "FightVideos",
    "ViolentVideos",
    "Abnormal",
)

# ============================================================
# EVALUATION
# ============================================================
PREFER_LEGACY_CACHE = True
LEGACY_CACHE_COLOR_ORDER = "BGR"
AUTO_SCALE_LEGACY_CACHE = True

EVAL_BATCH_SIZE = 8
NUM_WORKERS = 2

# Training/evaluation reference menggunakan FP32.
USE_AMP = False

BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 2026

ALLOW_BUILD_CACHE = True

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HELDOUT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("RUN_SEED          :", RUN_SEED)
print("NATIVE_T          :", NATIVE_T)
print("USE_TSM           :", USE_TSM)
print("SHIFT_DIV         :", SHIFT_DIV)
print("PRIMARY_VARIANT   :", PRIMARY_VARIANT)
print("RESOLVED_VARIANT  :", RESOLVED_VARIANT)
print("CHECKPOINT_PATH   :", CHECKPOINT_PATH)
print("EVAL_OUTPUT_DIR   :", EVAL_OUTPUT_DIR)

RUN_SEED          : 42
NATIVE_T          : 8
USE_TSM           : True
SHIFT_DIV         : 8
PRIMARY_VARIANT   : mobilenet_tsm
RESOLVED_VARIANT  : manual_override
CHECKPOINT_PATH   : /content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed42_v1/mobilenet_tsm/seed_42/best.pt
EVAL_OUTPUT_DIR   : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM/T8/seed_42


In [83]:
# ============================================================
# REPRODUCIBILITY / SAFETY
# ============================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
AMP_DTYPE = (
    torch.bfloat16
    if AMP_ENABLED and torch.cuda.is_bf16_supported()
    else torch.float16
)

def amp_context():
    if not AMP_ENABLED:
        return nullcontext()
    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def assert_finite_tensor(name: str, tensor: torch.Tensor) -> None:
    if not torch.isfinite(tensor).all():
        nan_count = int(torch.isnan(tensor).sum().item())
        inf_count = int(torch.isinf(tensor).sum().item())
        raise FloatingPointError(
            f"{name} mengandung NaN/Inf: nan={nan_count}, inf={inf_count}"
        )

def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )

set_global_seed(BOOTSTRAP_SEED)
print("Device:", DEVICE, "| AMP:", AMP_ENABLED)

Device: cuda | AMP: False


## 2. MobileNetV2 + Zhang Method-2 TSM

Evaluator mempertahankan implementasi training:

```python
for block in model.features:
    if isinstance(block, InvertedResidual)
    and block.use_res_connect
    and len(block.conv) >= 4:

        expansion = block.conv[0]

        block.conv[0] = nn.Sequential(
            expansion,
            TemporalShift(...)
        )
```

Jadi TSM dimasukkan **setelah pointwise expansion** pada residual
`InvertedResidual` yang eligible.

Forward clip:

```text
[B,T,3,64,64]
→ [B*T,3,64,64]
→ resize 224×224
→ MobileNetV2 + TSM
→ [B*T,2]
→ reshape [B,T,2]
→ mean logits sepanjang T
→ [B,2]
```

### T=1

Model **tetap mempunyai TSM wrappers**, tetapi:

```python
if self.n_segment == 1:
    return x
```

sehingga shift menjadi identity.

Ini sama dengan perilaku notebook training native T=1 dan memungkinkan satu
evaluator dipakai untuk T=1/2/4/8.

In [84]:
def select_segment_indices(
    num_segments: int,
) -> np.ndarray:
    if (
        CACHE_SEQUENCE_LENGTH
        % num_segments
        != 0
    ):
        raise ValueError(
            f"Cache length {CACHE_SEQUENCE_LENGTH} "
            f"tidak habis dibagi T={num_segments}."
        )

    segment_width = (
        CACHE_SEQUENCE_LENGTH
        // num_segments
    )

    offsets = []

    for segment_id in range(
        num_segments
    ):
        start = (
            segment_id
            * segment_width
        )

        # Training evaluation policy:
        # midpoint tiap temporal segment.
        offset = (
            segment_width
            // 2
        )

        offsets.append(
            start
            + offset
        )

    return np.asarray(
        offsets,
        dtype=np.int64,
    )


EVAL_SEGMENT_INDICES = select_segment_indices(
    NUM_SEGMENTS
)

EXPECTED_NATIVE_INDICES = {
    1: [8],
    2: [4, 12],
    4: [2, 6, 10, 14],
    8: [
        1,
        3,
        5,
        7,
        9,
        11,
        13,
        15,
    ],
}

if (
    EVAL_SEGMENT_INDICES
    .tolist()
    != EXPECTED_NATIVE_INDICES[
        NATIVE_T
    ]
):
    raise RuntimeError(
        "Native-T segment index audit failed: "
        f"{EVAL_SEGMENT_INDICES.tolist()} "
        f"!= {EXPECTED_NATIVE_INDICES[NATIVE_T]}"
    )

print(
    "Evaluation segment indices:",
    EVAL_SEGMENT_INDICES.tolist(),
)


class TemporalShift(nn.Module):
    def __init__(
        self,
        n_segment: int,
        n_div: int = SHIFT_DIV,
    ):
        super().__init__()

        self.n_segment = int(
            n_segment
        )

        self.n_div = int(
            n_div
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        if x.ndim != 4:
            raise ValueError(
                f"TemporalShift expects [B*T,C,H,W], "
                f"got {tuple(x.shape)}"
            )

        nt, c, h, w = (
            x.shape
        )

        # Exact T=1 behavior from training.
        if self.n_segment == 1:
            return x

        if (
            nt
            % self.n_segment
            != 0
        ):
            raise ValueError(
                f"B*T={nt} tidak habis dibagi "
                f"T={self.n_segment}."
            )

        batch_size = (
            nt
            // self.n_segment
        )

        x = x.view(
            batch_size,
            self.n_segment,
            c,
            h,
            w,
        )

        fold = (
            c
            // self.n_div
        )

        if fold == 0:
            return x.view(
                nt,
                c,
                h,
                w,
            )

        out = torch.zeros_like(
            x
        )

        # shift one channel group toward the previous position
        out[
            :,
            :-1,
            :fold,
        ] = x[
            :,
            1:,
            :fold,
        ]

        # shift second channel group toward the next position
        out[
            :,
            1:,
            fold:2 * fold,
        ] = x[
            :,
            :-1,
            fold:2 * fold,
        ]

        # remaining channels unchanged
        out[
            :,
            :,
            2 * fold:,
        ] = x[
            :,
            :,
            2 * fold:,
        ]

        return out.view(
            nt,
            c,
            h,
            w,
        )


def insert_zhang_method2_tsm(
    model: nn.Module,
    n_segment: int,
    n_div: int = SHIFT_DIV,
) -> int:
    inserted = 0

    for block in model.features:
        if not isinstance(
            block,
            InvertedResidual,
        ):
            continue

        if not block.use_res_connect:
            continue

        # Same eligibility rule as training:
        # residual block with expansion stage.
        if len(
            block.conv
        ) < 4:
            continue

        expansion = (
            block.conv[0]
        )

        block.conv[0] = nn.Sequential(
            expansion,
            TemporalShift(
                n_segment=n_segment,
                n_div=n_div,
            ),
        )

        inserted += 1

    return inserted


class VideoMobileNetTSMComparator(nn.Module):
    def __init__(
        self,
        num_segments: int,
        shift_div: int = SHIFT_DIV,
        pretrained: bool = False,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()

        self.use_tsm = True

        self.num_segments = int(
            num_segments
        )

        self.shift_div = int(
            shift_div
        )

        # Evaluation only:
        # entire trained state_dict will be loaded,
        # therefore pretrained weights are unnecessary here.
        self.base_model = mobilenet_v2(
            weights=None
        )

        in_features = int(
            self.base_model
            .classifier[1]
            .in_features
        )

        self.base_model.classifier[1] = nn.Linear(
            in_features,
            num_classes,
        )

        self.tsm_blocks = insert_zhang_method2_tsm(
            self.base_model,
            n_segment=self.num_segments,
            n_div=self.shift_div,
        )

    def forward(
        self,
        clip: torch.Tensor,
    ) -> torch.Tensor:
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,T,C,H,W], got {tuple(clip.shape)}"
            )

        (
            batch_size,
            seq_len,
            channels,
            height,
            width,
        ) = clip.shape

        if (
            seq_len
            != self.num_segments
        ):
            raise ValueError(
                f"Expected {self.num_segments} segments, "
                f"got {seq_len}"
            )

        if channels != 3:
            raise ValueError(
                f"Expected RGB 3 channels, got {channels}"
            )

        x = clip.reshape(
            batch_size
            * seq_len,
            channels,
            height,
            width,
        )

        x = F.interpolate(
            x,
            size=(
                BACKBONE_INPUT_SIZE,
                BACKBONE_INPUT_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        segment_logits = (
            self.base_model(
                x
            )
        )

        segment_logits = (
            segment_logits
            .view(
                batch_size,
                seq_len,
                NUM_CLASSES,
            )
        )

        # Same training consensus.
        return (
            segment_logits
            .mean(
                dim=1
            )
        )


# ------------------------------------------------------------
# Architecture audit independent of checkpoint.
# ------------------------------------------------------------
audit_model = VideoMobileNetTSMComparator(
    num_segments=NATIVE_T,
    shift_div=SHIFT_DIV,
    pretrained=False,
)

if audit_model.tsm_blocks <= 0:
    raise RuntimeError(
        "TSM insertion failed: no eligible residual block was wrapped."
    )

# Every inserted shift must have the requested T and shift_div.
shift_modules = [
    module
    for module
    in audit_model.modules()
    if isinstance(
        module,
        TemporalShift,
    )
]

if len(
    shift_modules
) != audit_model.tsm_blocks:
    raise RuntimeError(
        "TSM module count mismatch."
    )

for shift in shift_modules:
    if (
        shift.n_segment
        != NATIVE_T
    ):
        raise RuntimeError(
            "TemporalShift n_segment mismatch."
        )

    if (
        shift.n_div
        != SHIFT_DIV
    ):
        raise RuntimeError(
            "TemporalShift n_div mismatch."
        )

# Critical T=1 safety audit.
if NATIVE_T == 1:
    probe = torch.randn(
        2,
        32,
        8,
        8,
    )

    identity_shift = TemporalShift(
        n_segment=1,
        n_div=SHIFT_DIV,
    )

    with torch.inference_mode():
        shifted_probe = (
            identity_shift(
                probe
            )
        )

    if not torch.equal(
        probe,
        shifted_probe,
    ):
        raise RuntimeError(
            "T=1 TemporalShift harus exact identity."
        )

    print(
        "✅ T=1 TemporalShift identity audit PASS"
    )

print(
    "TSM blocks inserted:",
    audit_model.tsm_blocks,
)

del audit_model

Evaluation segment indices: [1, 3, 5, 7, 9, 11, 13, 15]
TSM blocks inserted: 10


## 3. Held-out cache dan native-T selection

Semua konfigurasi tetap menggunakan **shared 16-frame cache**:

```text
[N,16,64,64,3]
```

Tidak dibuat cache terpisah untuk T=1/2/4/8.

Pada inference:

```text
shared 16-frame cache
→ select deterministic indices sesuai NATIVE_T
→ ImageNet normalization
→ MobileNetV2+TSM
```

Jika cache UCF/Bus belum ada, raw clip akan dibuat menjadi cache RGB 16-frame
dengan pipeline yang sama dengan evaluator lainnya. Cache tersebut kemudian
dipakai ulang oleh semua seed dan model.

In [85]:
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm"
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {
        p.name: p
        for p in root.iterdir()
        if p.is_dir()
    }

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lower_map = {
        name.lower(): path
        for name, path in subdirs.items()
    }

    matches = []
    for alias in aliases:
        candidate = lower_map.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} "
            f"secara aman pada {root}.\n"
            f"Matches: {[p.name for p in matches]}\n"
            f"Available: {sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir,
            nonviolence_dir,
            NONVIOLENCE_ALIASES,
            "NonViolence",
        ),
        1: resolve_one_class_dir(
            raw_dir,
            violence_dir,
            VIOLENCE_ALIASES,
            "Violence",
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        videos = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not videos:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        print(
            f"{CLASS_NAMES[label]:12s}: {len(videos):5d} clips | {class_dir}"
        )

        records.extend(
            (p, label)
            for p in videos
        )

    labels = np.asarray(
        [label for _, label in records],
        dtype=np.int64,
    )

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus mempunyai kedua kelas: "
            "NonViolence dan Violence."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = CACHE_SEQUENCE_LENGTH,
    height: int = RAW_HEIGHT,
    width: int = RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(pos),
        )

        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )

        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)

def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    # Gunakan temporary memmap agar tidak menyimpan semua clip di RAM.
    n_requested = len(records)

    temp_path = cache_dir / f".{cache_name}_tmp_features.npy"

    if temp_path.exists():
        temp_path.unlink()

    temp = np.lib.format.open_memmap(
        temp_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_requested,
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_requested,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        temp[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    temp.flush()
    del temp

    if valid_count == 0:
        if temp_path.exists():
            temp_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    # Compact hanya sample valid ke final .npy.
    temp_read = np.load(
        temp_path,
        mmap_mode="r",
    )

    final = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 16

    for start in range(0, valid_count, chunk_size):
        end = min(start + chunk_size, valid_count)
        final[start:end] = temp_read[start:end]

    final.flush()
    del final, temp_read

    np.save(
        label_path,
        labels_buffer[:valid_count],
    )

    with open(
        paths_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_requested,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": CACHE_SEQUENCE_LENGTH,
                "height": RAW_HEIGHT,
                "width": RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if temp_path.exists():
        temp_path.unlink()

    print(
        f"Saved {feature_path} | "
        f"shape=({valid_count},{CACHE_SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3)"
    )
    print("Failed videos:", len(failed_paths))

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": "RGB",
        "cache_source": "built_rgb64_v2",
    }

def resolve_heldout_store(
    dataset_name: str,
    spec: Dict,
) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    rgb_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    rgb_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    rgb_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    # Sama dengan training: RGB-v2 diprioritaskan; legacy dipakai jika v2 tidak ada.
    if rgb_feature.exists() and rgb_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": rgb_feature,
            "label_path": rgb_label,
            "paths_path": rgb_paths if rgb_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if (
        PREFER_LEGACY_CACHE
        and legacy_feature.exists()
        and legacy_label.exists()
    ):
        print(
            f"✅ {dataset_name}: legacy cache "
            f"({LEGACY_CACHE_COLOR_ORDER}→RGB saat loading)"
        )
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache {dataset_name} tidak ada dan ALLOW_BUILD_CACHE=False."
        )

    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Cache dan raw folder tidak ditemukan untuk {dataset_name}.\n"
            f"Raw expected: {raw_dir}"
        )

    print(
        f"ℹ️ {dataset_name}: cache belum ada, dibuat dari {raw_dir}"
    )

    return build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

In [86]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(1, 3, 1, 1)


class HeldOutMobileNetSegmentDataset(Dataset):
    def __init__(
        self,
        feature_path: Path,
        label_path: Path,
        color_order: str,
        segment_indices: np.ndarray,
    ):
        self.feature_path = Path(
            feature_path
        )

        self.label_path = Path(
            label_path
        )

        self.features = np.load(
            self.feature_path,
            mmap_mode="r",
        )

        self.labels = np.load(
            self.label_path,
            mmap_mode="r",
        ).astype(
            np.int64
        )

        self.color_order = str(
            color_order
        ).upper()

        self.segment_indices = np.asarray(
            segment_indices,
            dtype=np.int64,
        )

        if len(self.features) != len(self.labels):
            raise ValueError(
                f"Feature/label mismatch: "
                f"{len(self.features)} vs {len(self.labels)}"
            )

        expected_shape = (
            CACHE_SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if (
            self.features.ndim != 5
            or tuple(
                self.features.shape[1:]
            ) != expected_shape
        ):
            raise ValueError(
                f"Unexpected cache shape {self.features.shape}; "
                f"expected [N,{CACHE_SEQUENCE_LENGTH},"
                f"{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        if tuple(
            self.segment_indices.shape
        ) != (
            NUM_SEGMENTS,
        ):
            raise ValueError(
                f"segment_indices shape "
                f"{self.segment_indices.shape}; "
                f"expected ({NUM_SEGMENTS},)"
            )

        if (
            self.segment_indices.min() < 0
            or self.segment_indices.max()
            >= CACHE_SEQUENCE_LENGTH
        ):
            raise ValueError(
                f"Invalid segment indices: "
                f"{self.segment_indices.tolist()}"
            )

        unique_labels = set(
            np.unique(
                self.labels
            ).tolist()
        )

        if unique_labels != {0, 1}:
            raise ValueError(
                f"Held-out dataset harus mempunyai label {{0,1}}; "
                f"ditemukan {sorted(unique_labels)}."
            )

        if self.color_order not in {
            "RGB",
            "BGR",
        }:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(
        self,
    ):
        return len(
            self.labels
        )

    def __getitem__(
        self,
        index: int,
    ):
        frames = np.asarray(
            self.features[
                index
            ],
            dtype=np.float32,
        )

        label = int(
            self.labels[
                index
            ]
        )

        if not np.isfinite(
            frames
        ).all():
            raise ValueError(
                f"NaN/Inf pada cache sample index={index}"
            )

        frame_min = float(
            frames.min()
        )

        frame_max = float(
            frames.max()
        )

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = (
                    frames
                    / 255.0
                )
            else:
                raise ValueError(
                    f"Invalid cache range "
                    f"[{frame_min},{frame_max}] "
                    f"at sample={index}"
                )

        if (
            float(
                frames.min()
            ) < -1e-4
            or float(
                frames.max()
            ) > 1.0001
        ):
            raise ValueError(
                f"Cache harus [0,1], ditemukan "
                f"[{float(frames.min())},"
                f"{float(frames.max())}] "
                f"at sample={index}"
            )

        if self.color_order == "BGR":
            frames = frames[
                ...,
                [2, 1, 0],
            ]

        selected = frames[
            self.segment_indices
        ]

        expected_selected_shape = (
            NUM_SEGMENTS,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if tuple(
            selected.shape
        ) != expected_selected_shape:
            raise RuntimeError(
                f"Selected clip shape "
                f"{selected.shape}; "
                f"expected {expected_selected_shape}"
            )

        clip = torch.from_numpy(
            np.ascontiguousarray(
                selected
            )
        ).permute(
            0,
            3,
            1,
            2,
        )

        clip = (
            clip
            - IMAGENET_MEAN
        ) / IMAGENET_STD

        return (
            clip,
            torch.tensor(
                label,
                dtype=torch.long,
            ),
            int(index),
        )


def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    path = Path(
        paths_path
    )

    if not path.exists():
        return None

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(
            file
        )

    paths = payload.get(
        "paths"
    )

    if not isinstance(
        paths,
        list,
    ):
        return None

    if len(paths) != expected_n:
        print(
            f"⚠️ paths.json count {len(paths)} != {expected_n}; "
            "video_path tidak dipakai."
        )
        return None

    return [
        str(path)
        for path in paths
    ]


def audit_dataset(
    dataset_name: str,
    store: Dict,
) -> Dict:
    dataset = HeldOutMobileNetSegmentDataset(
        store[
            "feature_path"
        ],
        store[
            "label_path"
        ],
        store[
            "color_order"
        ],
        segment_indices=EVAL_SEGMENT_INDICES,
    )

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=2,
    )

    audit_indices = np.unique(
        np.linspace(
            0,
            len(dataset) - 1,
            min(
                6,
                len(dataset),
            ),
        ).astype(int)
    )

    for idx in audit_indices:
        (
            clip,
            label,
            sample_idx,
        ) = dataset[
            int(idx)
        ]

        if tuple(
            clip.shape
        ) != (
            NUM_SEGMENTS,
            3,
            RAW_HEIGHT,
            RAW_WIDTH,
        ):
            raise RuntimeError(
                f"Invalid clip shape "
                f"at {sample_idx}: "
                f"{tuple(clip.shape)}"
            )

        assert_finite_tensor(
            f"{dataset_name}_sample_{sample_idx}",
            clip,
        )

    print(
        f"✅ {dataset_name}: "
        f"N={len(dataset)}, "
        f"NonViolence={counts[0]}, "
        f"Violence={counts[1]}, "
        f"T={NUM_SEGMENTS}, "
        f"indices={EVAL_SEGMENT_INDICES.tolist()}, "
        f"cache={store['cache_source']}"
    )

    return {
        "dataset": dataset_name,
        "n_total": int(
            len(dataset)
        ),
        "n_nonviolence": int(
            counts[0]
        ),
        "n_violence": int(
            counts[1]
        ),
        "native_t": int(
            NATIVE_T
        ),
        "segment_indices_0based": json.dumps(
            EVAL_SEGMENT_INDICES.tolist()
        ),
        "feature_path": str(
            store[
                "feature_path"
            ]
        ),
        "label_path": str(
            store[
                "label_path"
            ]
        ),
        "color_order_in_cache": store[
            "color_order"
        ],
        "cache_source": store[
            "cache_source"
        ],
    }


def make_loader(
    dataset: Dataset,
):
    generator = torch.Generator()

    generator.manual_seed(
        BOOTSTRAP_SEED
    )

    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
    )

## 4. Load dan audit `best.pt`

Evaluator wajib memakai **best checkpoint**, bukan `last.pt`.

Karena T tidak mengubah shape parameter learnable TSM, `strict=True` saja tidak
cukup untuk mendeteksi checkpoint T yang salah. Maka metadata checkpoint
diperiksa bila tersedia:

- `seed`
- `num_segments`
- `use_tsm`
- `shift_div`
- `validation_test_segment_indices`
- `model_variant`
- `temporal_shift_effective`

Setelah itu state-dict dimuat dengan `strict=True`.

Untuk T=1:
- `use_tsm` tetap `True`;
- TSM wrappers tetap ada;
- `temporal_shift_effective` seharusnya `False` bila metadata tersebut tersedia;
- operasi TemporalShift tetap identity.

In [87]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}"
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(
    checkpoint,
    dict,
):
    raise TypeError(
        "Checkpoint bukan dictionary dari notebook training."
    )

if (
    "model_state_dict"
    not in checkpoint
):
    raise KeyError(
        "Checkpoint tidak memiliki key 'model_state_dict'."
    )

checkpoint_config = checkpoint.get(
    "config",
    {},
)

mismatches = []

# ------------------------------------------------------------
# Seed
# ------------------------------------------------------------
if "seed" in checkpoint_config:
    actual = int(
        checkpoint_config[
            "seed"
        ]
    )

    if actual != RUN_SEED:
        mismatches.append(
            (
                "seed",
                actual,
                RUN_SEED,
            )
        )

# ------------------------------------------------------------
# Native T
# ------------------------------------------------------------
if (
    "num_segments"
    in checkpoint_config
):
    actual = int(
        checkpoint_config[
            "num_segments"
        ]
    )

    if actual != NATIVE_T:
        mismatches.append(
            (
                "num_segments",
                actual,
                NATIVE_T,
            )
        )

# ------------------------------------------------------------
# TSM flag
# ------------------------------------------------------------
if (
    "use_tsm"
    in checkpoint_config
):
    actual = bool(
        checkpoint_config[
            "use_tsm"
        ]
    )

    if actual is not True:
        mismatches.append(
            (
                "use_tsm",
                actual,
                True,
            )
        )

# ------------------------------------------------------------
# Shift division
# ------------------------------------------------------------
if (
    "shift_div"
    in checkpoint_config
):
    actual = int(
        checkpoint_config[
            "shift_div"
        ]
    )

    if actual != SHIFT_DIV:
        mismatches.append(
            (
                "shift_div",
                actual,
                SHIFT_DIV,
            )
        )

# ------------------------------------------------------------
# Shared cache length
# ------------------------------------------------------------
if (
    "cache_sequence_length"
    in checkpoint_config
):
    actual = int(
        checkpoint_config[
            "cache_sequence_length"
        ]
    )

    if (
        actual
        != CACHE_SEQUENCE_LENGTH
    ):
        mismatches.append(
            (
                "cache_sequence_length",
                actual,
                CACHE_SEQUENCE_LENGTH,
            )
        )

# ------------------------------------------------------------
# Evaluation temporal positions
# ------------------------------------------------------------
if (
    "validation_test_segment_indices"
    in checkpoint_config
):
    actual = list(
        checkpoint_config[
            "validation_test_segment_indices"
        ]
    )

    expected = (
        EVAL_SEGMENT_INDICES
        .tolist()
    )

    if actual != expected:
        mismatches.append(
            (
                "validation_test_segment_indices",
                actual,
                expected,
            )
        )

# ------------------------------------------------------------
# TSM effective flag:
# T=1 expected False, T>1 expected True.
# ------------------------------------------------------------
if (
    "temporal_shift_effective"
    in checkpoint_config
):
    actual = bool(
        checkpoint_config[
            "temporal_shift_effective"
        ]
    )

    expected = bool(
        NATIVE_T > 1
    )

    if actual != expected:
        mismatches.append(
            (
                "temporal_shift_effective",
                actual,
                expected,
            )
        )

# ------------------------------------------------------------
# Model variant
# ------------------------------------------------------------
if (
    "model_variant"
    in checkpoint_config
):
    actual_variant = str(
        checkpoint_config[
            "model_variant"
        ]
    )

    accepted_variants = {
        PRIMARY_VARIANT,
        ALTERNATE_VARIANT,
        f"mobilenet_tsm_native_T{NATIVE_T}",
    }

    if (
        actual_variant
        not in accepted_variants
    ):
        mismatches.append(
            (
                "model_variant",
                actual_variant,
                sorted(
                    accepted_variants
                ),
            )
        )

if mismatches:
    raise RuntimeError(
        "Checkpoint configuration mismatch:\n"
        + "\n".join(
            f"- {key}: actual={actual}, expected={expected}"
            for (
                key,
                actual,
                expected,
            )
            in mismatches
        )
    )

# ------------------------------------------------------------
# Build exact TSM architecture before loading state_dict.
# ------------------------------------------------------------
model = VideoMobileNetTSMComparator(
    num_segments=NATIVE_T,
    shift_div=SHIFT_DIV,
    pretrained=False,
).to(
    DEVICE
)

if not model.use_tsm:
    raise RuntimeError(
        "Evaluator harus menggunakan TSM."
    )

if model.tsm_blocks <= 0:
    raise RuntimeError(
        "No TSM blocks were inserted."
    )

load_result = (
    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )
)

model.eval()

# ------------------------------------------------------------
# Runtime shift-module audit
# ------------------------------------------------------------
runtime_shifts = [
    module
    for module
    in model.modules()
    if isinstance(
        module,
        TemporalShift,
    )
]

if (
    len(runtime_shifts)
    != model.tsm_blocks
):
    raise RuntimeError(
        "Runtime TSM module count mismatch."
    )

# T=1 identity must remain true even after checkpoint loading.
if NATIVE_T == 1:
    probe = torch.randn(
        2,
        32,
        8,
        8,
        device=DEVICE,
    )

    identity_module = TemporalShift(
        n_segment=1,
        n_div=SHIFT_DIV,
    ).to(
        DEVICE
    )

    with torch.inference_mode():
        shifted = identity_module(
            probe
        )

    if not torch.equal(
        probe,
        shifted,
    ):
        raise RuntimeError(
            "Loaded T=1 evaluator violates exact TSM identity."
        )

    del (
        probe,
        shifted,
        identity_module,
    )

# ------------------------------------------------------------
# End-to-end forward smoke test.
# ------------------------------------------------------------
with torch.inference_mode():
    dummy = torch.zeros(
        2,
        NUM_SEGMENTS,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )

    dummy_logits = model(
        dummy
    )

if (
    tuple(
        dummy_logits.shape
    )
    != (
        2,
        NUM_CLASSES,
    )
):
    raise RuntimeError(
        f"Unexpected output shape: "
        f"{tuple(dummy_logits.shape)}"
    )

assert_finite_tensor(
    "checkpoint_dummy_logits",
    dummy_logits,
)

print(
    "✅ MobileNetV2+TSM checkpoint audit PASS"
)

print(
    "Checkpoint        :",
    CHECKPOINT_PATH,
)

print(
    "Checkpoint variant:",
    RESOLVED_VARIANT,
)

print(
    "Native T          :",
    NATIVE_T,
)

print(
    "TSM active shift  :",
    NATIVE_T > 1,
)

print(
    "TSM blocks        :",
    model.tsm_blocks,
)

print(
    "Shift div         :",
    SHIFT_DIV,
)

print(
    "Indices           :",
    EVAL_SEGMENT_INDICES.tolist(),
)

print(
    "Epoch             :",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)

print(
    "Best val F1       :",
    checkpoint.get(
        "best_score",
        "unknown",
    ),
)

print(
    "Load result       :",
    load_result,
)

print(
    "Total parameters  :",
    sum(
        parameter.numel()
        for parameter
        in model.parameters()
    ),
)

del (
    dummy,
    dummy_logits,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ MobileNetV2+TSM checkpoint audit PASS
Checkpoint        : /content/drive/MyDrive/IVS/ViolenceDetection/Models/Comparators/mobilenet_tsm_Seed42_v1/mobilenet_tsm/seed_42/best.pt
Checkpoint variant: manual_override
Native T          : 8
TSM active shift  : True
TSM blocks        : 10
Shift div         : 8
Indices           : [1, 3, 5, 7, 9, 11, 13, 15]
Epoch             : 55
Best val F1       : 0.9504283965728274
Load result       : <All keys matched successfully>
Total parameters  : 2226434


## 5. Resolve dan audit tiga held-out dataset

In [88]:
heldout_stores = {}
dataset_audit_rows = []

for dataset_name, spec in HELDOUT_SPECS.items():
    print("\n" + "=" * 90)
    print("DATASET:", dataset_name)
    print("=" * 90)

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    heldout_stores[dataset_name] = store

    dataset_audit_rows.append(
        audit_dataset(
            dataset_name,
            store,
        )
    )

dataset_audit_df = pd.DataFrame(dataset_audit_rows)

dataset_audit_path = (
    EVAL_OUTPUT_DIR
    / "heldout_dataset_audit.csv"
)

dataset_audit_df.to_csv(
    dataset_audit_path,
    index=False,
)

display(dataset_audit_df)

print("Saved:", dataset_audit_path)


DATASET: RWF
✅ RWF: legacy cache (BGR→RGB saat loading)
✅ RWF: N=1998, NonViolence=1000, Violence=998, T=8, indices=[1, 3, 5, 7, 9, 11, 13, 15], cache=legacy

DATASET: UCF
✅ UCF: RGB-v2 cache
✅ UCF: N=100, NonViolence=50, Violence=50, T=8, indices=[1, 3, 5, 7, 9, 11, 13, 15], cache=rgb64_v2

DATASET: BV
✅ BV: RGB-v2 cache
✅ BV: N=1400, NonViolence=700, Violence=700, T=8, indices=[1, 3, 5, 7, 9, 11, 13, 15], cache=rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,native_t,segment_indices_0based,feature_path,label_path,color_order_in_cache,cache_source
0,RWF,1998,1000,998,8,"[1, 3, 5, 7, 9, 11, 13, 15]",/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,BGR,legacy
1,UCF,100,50,50,8,"[1, 3, 5, 7, 9, 11, 13, 15]",/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,rgb64_v2
2,BV,1400,700,700,8,"[1, 3, 5, 7, 9, 11, 13, 15]",/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,rgb64_v2


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM/T8/seed_42/heldout_dataset_audit.csv


## 6. Metrik dan prediction export

Per held-out dataset:

- Accuracy
- Balanced Accuracy
- Precision/Recall/F1 per kelas
- Macro-F1
- ROC-AUC
- Average Precision
- 95% bootstrap CI

`predictions.csv` menyimpan:

```text
dataset
variant
resolved_checkpoint_variant
seed
native_t
use_tsm
tsm_shift_effective
shift_div
segment_indices_0based
sample_index
sample_key
video_path
y_true
y_pred
is_correct
p_nonviolence
p_violence
```

Kolom `sample_key` sama dengan evaluator model lain sehingga siap untuk
McNemar dan paired bootstrap pada dataset/seed/T yang matching.

In [89]:
def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1],
            zero_division=0,
        )
    )

    result = {
        "n": int(len(y_true)),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision_nonviolence": float(precision[0]),
        "recall_nonviolence": float(recall[0]),
        "f1_nonviolence": float(f1[0]),
        "precision_violence": float(precision[1]),
        "recall_violence": float(recall[1]),
        "f1_violence": float(f1[1]),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(
            roc_auc_score(y_true, p_violence)
        )

        result["average_precision"] = float(
            average_precision_score(
                y_true,
                p_violence,
            )
        )
    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return result

@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for clips, labels, indices in tqdm(
        loader,
        desc="Inference",
        leave=False,
    ):
        clips = clips.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        assert_finite_tensor(
            "evaluation_input",
            clips,
        )

        with amp_context():
            logits = model(clips)

        assert_finite_tensor(
            "evaluation_logits",
            logits,
        )

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            indices.cpu().numpy().tolist()
        )
        all_labels.extend(
            labels.cpu().numpy().tolist()
        )
        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )
        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    sample_indices = np.asarray(
        all_indices,
        dtype=np.int64,
    )

    y_true = np.asarray(
        all_labels,
        dtype=np.int64,
    )

    y_pred = np.asarray(
        all_predictions,
        dtype=np.int64,
    )

    probs = np.asarray(
        all_probabilities,
        dtype=np.float64,
    )

    if len(y_true) == 0:
        raise RuntimeError(
            "Evaluation menghasilkan 0 predictions."
        )

    if probs.shape != (len(y_true), NUM_CLASSES):
        raise RuntimeError(
            f"Invalid probability shape {probs.shape}"
        )

    metrics = compute_classification_metrics(
        y_true,
        y_pred,
        probs[:, 1],
    )

    return (
        metrics,
        sample_indices,
        y_true,
        y_pred,
        probs,
    )

def bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    metric_name: str,
    n_bootstrap: int,
    seed: int,
    alpha: float = 0.05,
) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)

    n = len(y_true)
    values = []

    attempts = 0
    max_attempts = max(
        n_bootstrap * 5,
        1000,
    )

    while (
        len(values) < n_bootstrap
        and attempts < max_attempts
    ):
        attempts += 1

        idx = rng.integers(
            0,
            n,
            size=n,
        )

        yt = y_true[idx]
        yp = y_pred[idx]
        pp = p_violence[idx]

        if (
            metric_name in {"roc_auc", "average_precision"}
            and len(np.unique(yt)) < 2
        ):
            continue

        if metric_name == "accuracy":
            value = accuracy_score(yt, yp)
        elif metric_name == "f1_macro":
            value = f1_score(
                yt,
                yp,
                average="macro",
                zero_division=0,
            )
        elif metric_name == "f1_violence":
            value = f1_score(
                yt,
                yp,
                pos_label=1,
                zero_division=0,
            )
        elif metric_name == "roc_auc":
            value = roc_auc_score(yt, pp)
        elif metric_name == "average_precision":
            value = average_precision_score(
                yt,
                pp,
            )
        else:
            raise ValueError(metric_name)

        values.append(float(value))

    if not values:
        return float("nan"), float("nan")

    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )

def save_plots(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    title_prefix: str,
    output_dir: Path,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(5, 4)
    )

    image = ax.imshow(cm)

    for (row, col), value in np.ndenumerate(cm):
        ax.text(
            col,
            row,
            str(value),
            ha="center",
            va="center",
        )

    ax.set_xticks(
        [0, 1],
        CLASS_NAMES,
    )
    ax.set_yticks(
        [0, 1],
        CLASS_NAMES,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(
        f"{title_prefix} — Confusion Matrix"
    )
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(
        output_dir / "confusion_matrix.png",
        dpi=180,
    )
    plt.close(fig)

    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(
            y_true,
            p_violence,
        )

        auc_value = roc_auc_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(
            figsize=(5, 4)
        )
        ax.plot(
            fpr,
            tpr,
            label=f"AUC={auc_value:.4f}",
        )
        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
        )
        ax.set_xlabel(
            "False Positive Rate"
        )
        ax.set_ylabel(
            "True Positive Rate"
        )
        ax.set_title(
            f"{title_prefix} — ROC"
        )
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "roc_curve.png",
            dpi=180,
        )
        plt.close(fig)

        pr_precision, pr_recall, _ = (
            precision_recall_curve(
                y_true,
                p_violence,
            )
        )

        ap_value = average_precision_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(
            figsize=(5, 4)
        )
        ax.plot(
            pr_recall,
            pr_precision,
            label=f"AP={ap_value:.4f}",
        )
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(
            f"{title_prefix} — Precision-Recall"
        )
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "precision_recall_curve.png",
            dpi=180,
        )
        plt.close(fig)

## 7. Jalankan held-out evaluation

Cell ini hanya melakukan forward inference pada checkpoint MobileNetV2 tanpa
TSM yang sudah dibekukan.

In [90]:
results = []

ci_metric_names = [
    "accuracy",
    "f1_macro",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

for dataset_position, (
    dataset_name,
    store,
) in enumerate(
    heldout_stores.items()
):
    print("\n" + "=" * 90)
    print(
        f"EVALUATE {dataset_name} | "
        f"MobileNetV2+TSM T={NATIVE_T} seed={RUN_SEED}"
    )
    print("=" * 90)

    dataset = HeldOutMobileNetSegmentDataset(
        store["feature_path"],
        store["label_path"],
        store["color_order"],
        segment_indices=EVAL_SEGMENT_INDICES,
    )

    loader = make_loader(dataset)

    (
        metrics,
        sample_indices,
        y_true,
        y_pred,
        probs,
    ) = evaluate_loader(
        model,
        loader,
        DEVICE,
    )

    output_dir = (
        EVAL_OUTPUT_DIR
        / dataset_name
    )
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    source_paths = load_paths_if_available(
        store.get("paths_path"),
        expected_n=len(dataset),
    )

    if source_paths is None:
        sample_keys = [
            f"{dataset_name}::{int(index)}"
            for index in sample_indices
        ]
        video_paths = [
            ""
            for _ in sample_indices
        ]
    else:
        video_paths = [
            source_paths[int(index)]
            for index in sample_indices
        ]
        sample_keys = [
            Path(path).as_posix()
            for path in video_paths
        ]

    prediction_df = pd.DataFrame(
        {
            "dataset": dataset_name,
            "variant": PRIMARY_VARIANT,
            "resolved_checkpoint_variant": RESOLVED_VARIANT,
            "seed": RUN_SEED,
            "native_t": NATIVE_T,
            "use_tsm": True,
            "tsm_shift_effective": bool(NATIVE_T > 1),
            "shift_div": SHIFT_DIV,
            "segment_indices_0based": json.dumps(
                EVAL_SEGMENT_INDICES.tolist()
            ),
            "sample_index": sample_indices,
            "sample_key": sample_keys,
            "video_path": video_paths,
            "y_true": y_true,
            "y_pred": y_pred,
            "is_correct": (
                y_true == y_pred
            ).astype(np.int64),
            "p_nonviolence": probs[:, 0],
            "p_violence": probs[:, 1],
        }
    )

    # Audit unik agar paired comparison/McNemar aman.
    if prediction_df["sample_key"].duplicated().any():
        duplicated = prediction_df.loc[
            prediction_df["sample_key"].duplicated(),
            "sample_key",
        ].head(10).tolist()

        raise RuntimeError(
            f"Duplicate sample_key pada {dataset_name}: {duplicated}"
        )

    prediction_path = (
        output_dir
        / "predictions.csv"
    )

    prediction_df.to_csv(
        prediction_path,
        index=False,
    )

    for metric_position, metric_name in enumerate(
        ci_metric_names
    ):
        ci_seed = (
            BOOTSTRAP_SEED
            + dataset_position * 100
            + metric_position
        )

        low, high = bootstrap_ci(
            y_true=y_true,
            y_pred=y_pred,
            p_violence=probs[:, 1],
            metric_name=metric_name,
            n_bootstrap=BOOTSTRAP_SAMPLES,
            seed=ci_seed,
        )

        metrics[
            f"{metric_name}_ci95_low"
        ] = low

        metrics[
            f"{metric_name}_ci95_high"
        ] = high

    metrics.update(
        {
            "variant": PRIMARY_VARIANT,
            "resolved_checkpoint_variant": RESOLVED_VARIANT,
            "native_t": NATIVE_T,
            "use_tsm": True,
            "tsm_shift_effective": bool(NATIVE_T > 1),
            "shift_div": SHIFT_DIV,
            "segment_indices_0based": json.dumps(EVAL_SEGMENT_INDICES.tolist()),
            "seed": RUN_SEED,
            "dataset": dataset_name,
            "checkpoint_path": str(
                CHECKPOINT_PATH
            ),
            "best_epoch": checkpoint.get(
                "epoch",
                np.nan,
            ),
            "best_val_f1_macro": checkpoint.get(
                "best_score",
                np.nan,
            ),
            "cache_source": store[
                "cache_source"
            ],
        }
    )

    results.append(metrics)

    save_plots(
        y_true=y_true,
        y_pred=y_pred,
        p_violence=probs[:, 1],
        title_prefix=(
            f"MobileNetV2+TSM T{NATIVE_T} seed {RUN_SEED} — {dataset_name}"
        ),
        output_dir=output_dir,
    )

    print(
        f"N={metrics['n']} | "
        f"AP={metrics['average_precision']:.4f} | "
        f"AUC={metrics['roc_auc']:.4f} | "
        f"Macro-F1={metrics['f1_macro']:.4f}"
    )

    del dataset, loader, probs

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

heldout_metrics_df = pd.DataFrame(results)

metrics_output = (
    EVAL_OUTPUT_DIR
    / f"heldout_metrics_mobilenet_tsm_T{NATIVE_T}_seed_{RUN_SEED}.csv"
)

heldout_metrics_df.to_csv(
    metrics_output,
    index=False,
)

display_columns = [
    "dataset",
    "n",
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "precision_violence",
    "recall_violence",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

print("\n=== HELD-OUT RESULTS ===")
display(
    heldout_metrics_df[
        display_columns
    ].round(4)
)

print("Saved:", metrics_output)


EVALUATE RWF | MobileNetV2+TSM T=8 seed=42


Inference:   0%|          | 0/250 [00:00<?, ?it/s]

N=1998 | AP=0.7237 | AUC=0.7183 | Macro-F1=0.6552

EVALUATE UCF | MobileNetV2+TSM T=8 seed=42


Inference:   0%|          | 0/13 [00:00<?, ?it/s]

N=100 | AP=0.7674 | AUC=0.7566 | Macro-F1=0.7097

EVALUATE BV | MobileNetV2+TSM T=8 seed=42


Inference:   0%|          | 0/175 [00:00<?, ?it/s]

N=1400 | AP=0.6111 | AUC=0.6233 | Macro-F1=0.4389

=== HELD-OUT RESULTS ===


,dataset,n,accuracy,balanced_accuracy,f1_macro,precision_violence,recall_violence,f1_violence,roc_auc,average_precision
0,RWF,1998,0.6567,0.6566,0.6552,0.6793,0.5922,0.6328,0.7183,0.7237
1,UCF,100,0.7100,0.7100,0.7097,0.6981,0.7400,0.7184,0.7566,0.7674
2,BV,1400,0.5336,0.5336,0.4389,0.6880,0.1229,0.2085,0.6233,0.6111


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM/T8/seed_42/heldout_metrics_mobilenet_tsm_T8_seed_42.csv


## 8. Simpan evaluation manifest

Manifest mencatat:
- checkpoint;
- seed;
- native T;
- TSM active/identity status;
- shift divisor;
- deterministic segment indices;
- consensus;
- shared held-out cache provenance.

In [91]:
manifest = {
    "task": "heldout_evaluation_only",
    "training_performed": False,
    "model_family": "MobileNetV2+TSM",
    "uses_tsm": True,
    "tsm_method": "Zhang-style Method-2 insertion after expansion in eligible residual InvertedResidual blocks",
    "tsm_shift_effective": bool(
        NATIVE_T > 1
    ),
    "t1_behavior": (
        "TemporalShift identity; TSM wrappers retained"
        if NATIVE_T == 1
        else None
    ),
    "shift_div": SHIFT_DIV,
    "tsm_blocks": model.tsm_blocks,
    "variant_requested": PRIMARY_VARIANT,
    "variant_resolved": RESOLVED_VARIANT,
    "checkpoint_seed": RUN_SEED,
    "native_t": NATIVE_T,
    "num_segments": NUM_SEGMENTS,
    "checkpoint_path": str(
        CHECKPOINT_PATH
    ),
    "checkpoint_epoch": checkpoint.get(
        "epoch"
    ),
    "checkpoint_best_val_f1_macro": checkpoint.get(
        "best_score"
    ),
    "device": str(
        DEVICE
    ),
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "cache_sequence_length": CACHE_SEQUENCE_LENGTH,
    "evaluation_segment_indices_0based": (
        EVAL_SEGMENT_INDICES
        .tolist()
    ),
    "native_t_expected_indices": (
        EXPECTED_NATIVE_INDICES[
            NATIVE_T
        ]
    ),
    "raw_resolution": [
        RAW_HEIGHT,
        RAW_WIDTH,
    ],
    "backbone_input_size": [
        BACKBONE_INPUT_SIZE,
        BACKBONE_INPUT_SIZE,
    ],
    "consensus": "average_of_segment_logits",
    "normalization": "ImageNet",
    "use_amp": USE_AMP,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "heldout_datasets": list(
        HELDOUT_SPECS.keys()
    ),
    "checkpoint_config": checkpoint_config,
}

manifest_path = (
    EVAL_OUTPUT_DIR
    / "evaluation_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        indent=2,
        default=str,
    )

print(
    "Manifest:",
    manifest_path,
)

print(
    "\n✅ MobileNetV2+TSM held-out evaluation selesai."
)

print(
    "✅ Tidak ada training/backpropagation."
)

Manifest: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/MobileNetV2_TSM/T8/seed_42/evaluation_manifest.json

✅ MobileNetV2+TSM held-out evaluation selesai.
✅ Tidak ada training/backpropagation.


# Evaluasi konfigurasi berikutnya

Untuk seed lain pada T yang sama:

```python
RUN_SEED = 42
NATIVE_T = 8
```

cukup ubah:

```python
RUN_SEED = 123
```

Untuk T lain, ubah `NATIVE_T` juga:

```python
RUN_SEED = 123
NATIVE_T = 1
```

atau:

```python
RUN_SEED = 42
NATIVE_T = 4
```

Jangan ubah antar-run:
- `SHIFT_DIV = 8`
- shared cache 16-frame
- deterministic segment-center policy
- ImageNet normalization
- TSM insertion rule
- average-logit consensus

Satu notebook ini mencakup:

```text
4 nilai T × 3 seed = 12 evaluation runs
```

### Penting untuk T=1

Jangan menonaktifkan `USE_TSM` dan jangan mengganti model menjadi evaluator
MobileNetV2 tanpa TSM. Checkpoint T=1 tetap berasal dari model dengan wrapper
TSM; hanya operasi TemporalShift-nya yang identity.